In [1]:
# Useful for debugging
%load_ext autoreload
%autoreload 2

In [2]:
import os
import torch
import numpy as np
import json
import pandas as pd
import seaborn as sns
from datetime import datetime

In [3]:
# BoTorch models and transforms
from botorch.models import SingleTaskGP, ModelListGP
from botorch.models.transforms import Normalize, Standardize

from botorch.fit import fit_gpytorch_mll
fit_gpytorch_mll = fit_gpytorch_mll

# GPyTorch MLL
from gpytorch.mlls import ExactMarginalLogLikelihood

# Multi-objective Monte Carlo acquisition functions
from botorch.acquisition.multi_objective import (
    qExpectedHypervolumeImprovement,
    qLogNoisyExpectedHypervolumeImprovement,
)

# BoTorch utilities
from botorch.utils.multi_objective.box_decompositions import FastNondominatedPartitioning
from botorch.optim import optimize_acqf
from botorch.utils.multi_objective.pareto import is_non_dominated
from botorch.utils.multi_objective.hypervolume import Hypervolume

# For Parallel Executions
from joblib import Parallel, delayed
from concurrent.futures import ThreadPoolExecutor

/home/cspark/Work/simulation_codes-working/miniforge3/envs/linac-opt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Nicer plotting
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12,8)
%config InlineBackend.figure_format = 'retina'

In [5]:
from astra import Astra
# load astra and generator binaries
%env ASTRA_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra
%env GENERATOR_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/generator
!echo $ASTRA_BIN
!echo $GENERATOR_BIN

env: ASTRA_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra
env: GENERATOR_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/generator
/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra
/home/cspark/Work/simulation_codes-working/lume-astra/bin/generator


In [6]:
from run_astra import *
from mobo_utils import *
from utils import *
from file_io import *
from plot_utils import *

In [7]:
# Set default tensor type to double for BoTorch
torch.set_default_dtype(torch.double)

In [ ]:
# ── Simulation Configuration Summary ─────────────────────────────────────────
# Displays key parameters for this notebook run in tabular format.
# Defaults reflect original phase2_mobo configuration.
# 'This Session' shows any overrides applied for the current run.

import pandas as pd

# --------------------------------------------------------------------------
# Notebook-level overrides (edit these to reconfigure a run)
# --------------------------------------------------------------------------
_N_ITERATIONS  = 300   # default: 300
_BATCH_SIZE    = 8     # default:   8 (q)
_INIT_SAMPLES  = 16    # default:  16
_NUM_WORKERS   = 12    # default:  12 (ThreadPoolExecutor)
_SEED          = 42    # default:  42
_ACQ_FN        = "qLogNEHVI"   # default: qLogNEHVI  (set in cell 9)
_REF_POINT     = [1.5e-6, 1.5e-6, 1.5e6]  # default reference point [m·rad, m·rad, eV]
_ASTRA_BIN     = "/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra"

# --------------------------------------------------------------------------
# Build summary table
# --------------------------------------------------------------------------
_rows = [
    ("ASTRA binary",      "./bin/astra (project local)",
                           _ASTRA_BIN,                       "$ASTRA_BIN env var"),
    ("BO iterations",     "300",   str(_N_ITERATIONS),        "total iterations"),
    ("Batch size (q)",    "8",     str(_BATCH_SIZE),          "candidates/iter"),
    ("Initial samples",  "16",     str(_INIT_SAMPLES),        "Sobol random init"),
    ("Parallel workers", "12",     str(_NUM_WORKERS),         "ThreadPoolExecutor"),
    ("Random seed",      "42",     str(_SEED),                "reproducibility"),
    ("Surrogate model",  "ModelListGP",  "ModelListGP",       "3 ind. Matérn-5/2 ARD GPs"),
    ("Acquisition fn",   "qLogNEHVI",   _ACQ_FN,             "set in Acq. cell"),
    ("Reference point",  "[1.5e-6, 1.5e-6, 1.5e6]",
                           str(_REF_POINT),                   "[m·rad, m·rad, eV]"),
    ("Objectives",       "ex, ey, sigma_E",  "ex, ey, sigma_E", "minimise all 3"),
    ("Design variables", "6D", "6D", "sol, Q1, Q2, phi_gun, phi12, phi34"),
]

_df = pd.DataFrame(_rows, columns=["Parameter", "Default", "This Session", "Notes"])
_df["Changed"] = _df.apply(
    lambda r: "✔ reconfigured" if r["Default"] != r["This Session"] else "", axis=1
)

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║   Phase 2 · MOBO · Simulation Configuration Summary                  ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
display(_df.to_string(index=False))
try:
    from IPython.display import display as _disp
    _disp(_df.style
         .set_caption("Phase 2 MOBO — Configuration Summary")
         .applymap(lambda v: "background-color:#ffeeba;font-weight:bold" if v == "✔ reconfigured" else "",
                   subset=["Changed"])
         .set_properties(**{"text-align": "left"})
         .hide(axis="index")
    )
except Exception:
    pass


In [8]:
# Wrapper for concurrent evaluation
executor = ThreadPoolExecutor(max_workers=12)

In [9]:
# --- Checkpointing Configuration ---
CHECKPOINT_FILE = "results_notebooks/phase2_mobo/checkpoints/mobo_checkpoint.pt"

In [10]:
# -----------------------------------------------------------
# 0. Acquisition Function Selection (Choose one here)
# -----------------------------------------------------------
# Choose your desired acquisition function by uncommenting one line:
#acquisition_mode = 'qEHVI'
acquisition_mode = 'qLogNEHVI'
# -----------------------------------------------------------

In [ ]:
# -------------------------
# 3. Initialize Training Data
# -------------------------
print("Initializing training data...")

# Attempt to load checkpoint
checkpoint = load_checkpoint()
#ratio_list = [0.10, 0.10, 0.10, 0.10, 0.10, 0.10]
ratio_list = [0.50, 0.50, 0.50, 0.50, 0.50, 0.50]

if checkpoint:
    start_iteration = checkpoint['iteration'] + 1
    train_X = checkpoint['train_X']
    train_Y = checkpoint['train_Y']
    train_feas_mask = checkpoint['train_feas_mask']
    hypervolumes = checkpoint['hypervolumes']
    train_constraints_list = checkpoint['train_constraints_list']
    # Ensure acquisition_mode matches if loaded
    if acquisition_mode != checkpoint['acquisition_mode']:
        print(f"Warning: Acquisition mode changed from {checkpoint['acquisition_mode']} to {acquisition_mode}. "
              "Ensure this is intentional when resuming.")
    
    # Reload Astra config for initial parameters if needed (assuming it's consistent)
    A = Astra('astra.in')
    A.timeout = None
    A.verbose = False
    A.run()
    init_parameters = [
        A['solenoid:maxb(1)'], A['quadrupole:q_grad(1)'], A['quadrupole:q_grad(2)'],
        A['cavity:phi(1)'], A['cavity:phi(2)'], A['cavity:phi(4)']
    ]
    
    param_bounds_list = []
    for val, ratio in zip(init_parameters, ratio_list):
        lower_val = val * (1 - ratio)
        upper_val = val * (1 + ratio)
        param_bounds_list.append([min(lower_val, upper_val), max(lower_val, upper_val)])
    bounds = torch.tensor(param_bounds_list, dtype=torch.double).T
    input_transform = Normalize(d=bounds.shape[1], bounds=bounds)

else:
    start_iteration = 0
    A = Astra('astra.in')
    A.timeout = None
    A.verbose = False
    A.run()

    # Extract initial parameters from the Astra object (6 independent parameters)
    init_parameters = [
        A['solenoid:maxb(1)'],
        A['quadrupole:q_grad(1)'],
        A['quadrupole:q_grad(2)'],
        A['cavity:phi(1)'],
        A['cavity:phi(2)'], # Use phi(2) as the initial value for common_phi_2_3
        A['cavity:phi(3)'], # Common phase for cavity 2 & 3
        A['cavity:phi(4)'],  # Use phi(4) as the initial value for common_phi_4_5
        A['cavity:phi(5)'] # Common phase for cavity 4 & 5
    ]

    # Define bounds as a ratio around initial parameters
    num_initial_samples = 16 # Changed from 10 to 15 for better initial exploration
    # Use a list of ratios for each parameter for more granular control
    param_bounds_list = []
    for val, ratio in zip(init_parameters, ratio_list):
        lower_val = val * (1 - ratio)
        upper_val = val * (1 + ratio)
        # Ensure lower bound is always <= upper bound, especially for negative initial values
        param_bounds_list.append([min(lower_val, upper_val), max(lower_val, upper_val)])

    # Convert bounds to a PyTorch tensor, transposed
    bounds = torch.tensor(param_bounds_list, dtype=torch.double).T 

    # Define input_transform using Normalize based on the bounds
    input_transform = Normalize(d=bounds.shape[1], bounds=bounds)

    # Generate initial random samples within bounds
    train_X_list = []
    for _ in range(num_initial_samples):
        sample = [np.random.uniform(low, high) for (low, high) in param_bounds_list]
        train_X_list.append(torch.tensor(sample, dtype=torch.double))
    train_X = torch.stack(train_X_list) 

    # Evaluate initial samples in parallel using ThreadPoolExecutor
    print(f"Evaluating {num_initial_samples} initial samples in parallel...")
    eval_results = list(executor.map(evaluate_objective, train_X))
    
    # Unpack results
    train_Y_list, train_feas_list, initial_constraints_list = zip(*eval_results)
    train_Y = torch.stack(train_Y_list)  # Objectives (negated)
    train_feas_mask = torch.stack(train_feas_list) # Feasibility mask
    train_constraints_list = list(initial_constraints_list)
    # train_constraints_df = pd.DataFrame(train_constraints_list) # This is for plotting, not needed for core state

print(f"Initial/Loaded training data evaluated. Feasible samples: {train_feas_mask.sum().item()} / {train_X.shape[0]}")

Initializing training data...
No checkpoint found. Starting new optimization.
Evaluating 16 initial samples in parallel...
Running simulation with parameters: [0.2370714119254546, 1.8634484103801348, -3.4112629360817026, 53.17977406787008, -37.306435128637865, -51.51331017457584]
Running simulation with parameters: [0.21135413868090158, 1.3853612992137325, -4.0160736700748085, 49.21371439697889, -32.191488517635065, -30.044278425338163]
Running simulation with parameters: [0.15563273773565287, 1.8924052064015122, -2.3484573278164285, 28.960221328414754, -44.70970413184233, -57.54013272908231]
Running simulation with parameters: [0.25700399998415996, 1.5967591295325994, -3.3784817960438405, 30.099399070589598, -38.014334990171065, -20.120512977446126]
Running simulation with parameters: [0.15767113899456042, 1.471592201233691, -3.9585678759164193, 26.890030256629064, -26.792945950738144, -27.181913619376886]
Running simulation with parameters: [0.12644345942262264, 0.9286831254110598, -

In [ ]:
n_iterations = 20 # Production Bayesian Optimization iterations


In [ ]:
# Save final results (negated objectives)
save_results(train_X, train_Y)

In [ ]:
# --- Plot Hypervolume vs Iteration Plot
print("Generating Hypervolume progress plot...")
plot_hypervolume(hypervolumes, n_iterations, start_iteration)

In [ ]:
# Generate 2D projection plots of the objective space
print("\nGenerating final Pareto front 2D projection plots...")
# Updated call to pass only train_Y as the function now calculates its own plotting data
plot_pareto_objective_space(train_Y)

## Visualization & Analysis

Comprehensive post-run plotting for Phase 2 MOBO.

> **Note**: Run the BO loop above before executing these cells.

In [ ]:
# ── Plotting imports ─────────────────────────────────────────────────────────
from mobo_linac.plotting import (
    plot_hypervolume_progress,
    plot_pareto_front,
    plot_pareto_front_3d,
    plot_objective_evolution,
    plot_best_so_far,
    plot_feasibility_rate,
    plot_constraint_diagnostics,
    plot_constraint_violins,
    plot_design_variable_heatmap,
    plot_parallel_coordinates,
    plot_gp_surrogate_slice,
)
from pathlib import Path

# Build EvaluationResult list from legacy train_X / train_Y tensors if available
# (Phase 2 stores raw tensors rather than EvaluationResult objects)
# If you have EvaluationResult objects, assign them to `results` directly.
results = []   # replace with actual EvaluationResult list if available

_figures_dir = Path(CHECKPOINT_FILE).parent.parent / "figures"
_figures_dir.mkdir(parents=True, exist_ok=True)
print(f"Saving figures to: {_figures_dir}")


### Hypervolume Progress

In [ ]:
# Plot hypervolume from the tracked list
if hypervolumes:
    hv_df = pd.DataFrame({"iteration": range(1, len(hypervolumes)+1),
                          "feasible_hypervolume": hypervolumes})
    fig = plot_hypervolume_progress(hv_df,
                                   output_path=_figures_dir / "hypervolume_progress.png")
    plt.show()


### Objective Space — 2D Projections

In [ ]:
# 2D Pareto projections (requires EvaluationResult list)
if results:
    fig = plot_pareto_front(results,
                           output_path=_figures_dir / "pareto_2d.png")
    plt.show()
else:
    # Fallback: plot directly from train_Y tensor (negated model-space → physical)
    # train_Y columns: [−ε_x, −ε_y, −σ_E]  (model space maximisation)
    phys_Y = -train_Y.cpu().numpy()   # back to positive physical values
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    labels = [r'$\varepsilon_{n,x}$ [m·rad]',
              r'$\varepsilon_{n,y}$ [m·rad]',
              r'$\sigma_E$ [eV]']
    pairs = [(0,1), (2,0), (2,1)]
    for ax, (xi, yi) in zip(axes, pairs):
        ax.scatter(phys_Y[:, xi], phys_Y[:, yi], c='steelblue', s=25, alpha=0.7)
        ax.set_xlabel(labels[xi], fontsize=11)
        ax.set_ylabel(labels[yi], fontsize=11)
        ax.grid(True, linestyle=':', alpha=0.6)
    fig.suptitle('Objective Space — train_Y (Phase 2)', fontsize=14)
    fig.tight_layout()
    plt.savefig(_figures_dir / 'pareto_2d_raw.png', dpi=200)
    plt.show()


### 3D Pareto Scatter

In [ ]:
# 3D scatter — EvaluationResult list required
if results:
    fig = plot_pareto_front_3d(results, elev=25, azim=45,
                               output_path=_figures_dir / "pareto_3d.png")
    plt.show()
else:
    # Fallback from raw tensor
    from mpl_toolkits.mplot3d import Axes3D  # noqa
    phys_Y = -train_Y.cpu().numpy()
    fig = plt.figure(figsize=(9, 7))
    ax3 = fig.add_subplot(111, projection='3d')
    sc = ax3.scatter(phys_Y[:,0]*1e6, phys_Y[:,1]*1e6, phys_Y[:,2]*1e-6,
                     c=phys_Y[:,2]*1e-6, cmap='plasma', s=35, alpha=0.8)
    fig.colorbar(sc, ax=ax3, shrink=0.6, label=r'$\sigma_E$ [MeV]')
    ax3.set_xlabel(r'$\varepsilon_{n,x}$ [mm·mrad]')
    ax3.set_ylabel(r'$\varepsilon_{n,y}$ [mm·mrad]')
    ax3.set_zlabel(r'$\sigma_E$ [MeV]')
    ax3.set_title('3D Pareto — Phase 2')
    plt.savefig(_figures_dir / 'pareto_3d_raw.png', dpi=200)
    plt.show()


### Objective Evolution & Best-So-Far

In [ ]:
if results:
    fig = plot_objective_evolution(results,
                                  output_path=_figures_dir / "objective_evolution.png")
    plt.show()
    fig = plot_best_so_far(results,
                          output_path=_figures_dir / "best_so_far.png")
    plt.show()
else:
    # Fallback: rolling minimum from tensor
    phys_Y = -train_Y.cpu().numpy()
    fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
    labels = [r'$\varepsilon_{n,x}$ [m·rad]',
              r'$\varepsilon_{n,y}$ [m·rad]',
              r'$\sigma_E$ [eV]']
    colors = ['steelblue', 'darkorange', 'seagreen']
    for i, (ax, lbl, col) in enumerate(zip(axes, labels, colors)):
        vals = phys_Y[:, i]
        ax.plot(vals, color=col, linewidth=1.4, alpha=0.8)
        ax.plot(pd.Series(vals).cummin(), color=col, linestyle='--',
                linewidth=2, label='Best so far')
        ax.set_ylabel(lbl, fontsize=11)
        ax.grid(True, linestyle=':', alpha=0.6)
        ax.legend(fontsize=9)
    axes[-1].set_xlabel('Evaluation Index', fontsize=12)
    fig.suptitle('Objective Evolution — Phase 2', fontsize=14)
    fig.tight_layout()
    plt.savefig(_figures_dir / 'objective_evolution_raw.png', dpi=200)
    plt.show()


### Feasibility Rate

In [ ]:
if results:
    fig = plot_feasibility_rate(results, window=10,
                               output_path=_figures_dir / "feasibility_rate.png")
    plt.show()


### Constraint Diagnostics & Violin Plots

In [ ]:
if results:
    fig = plot_constraint_diagnostics(results,
                                     output_path=_figures_dir / "constraint_diagnostics.png")
    plt.show()
    fig = plot_constraint_violins(results,
                                 output_path=_figures_dir / "constraint_violins.png")
    plt.show()
else:
    # Fallback — from legacy constraint DataFrame
    final_df = pd.DataFrame(train_constraints_list)
    plot_all_constraints(final_df, train_feas_mask)


### Design Variable Analysis

In [ ]:
if results:
    fig = plot_design_variable_heatmap(results, feasible_only=True,
                                      output_path=_figures_dir / "design_var_heatmap.png")
    plt.show()
    fig = plot_parallel_coordinates(results, color_by='norm_emit_x_m_rad',
                                   feasible_only=True, n_lines=200,
                                   output_path=_figures_dir / "parallel_coordinates.png")
    plt.show()


### GP Surrogate Slice

Queries the fitted GP posterior along a 1D sweep of each design variable.

In [ ]:
# Requires `model` (ModelListGP) and `bounds` from the BO loop above.
# Uncomment the block below after fitting the model.
#
# from mobo_linac.plotting import plot_gp_surrogate_slice
# import torch
#
# fixed_x = bounds.mean(dim=0)   # use midpoint as nominal reference
# for dim in range(6):
#     for obj_idx in range(3):   # ε_x, ε_y, σ_E
#         fig = plot_gp_surrogate_slice(
#             model, bounds, fixed_x,
#             dim=dim, obj_idx=obj_idx,
#             output_path=_figures_dir / f'gp_slice_dim{dim}_obj{obj_idx}.png',
#         )
#         plt.show()
print('GP surrogate slice: uncomment the block above and supply the fitted model.')
